In [2]:
import pandas as pd
invoices = pd.read_csv('../data/raw/invoices.csv')
invoices.head()


,invoice_id,so_id,customer_id,branch_id,invoice_date,due_date,total_order_value,total_gst_amount,grand_total,payment_status
0,INV-325361,SO-990591,C0070,AHM001,2022-05-28,2022-06-27,1345720,374439.6,1720159.6,Unpaid
1,INV-568476,SO-431510,C0031,KOL001,2022-05-08,2022-06-07,2194850,609423.0,2804273.0,Unpaid
2,INV-786913,SO-495859,C0013,AHM001,2023-10-02,2023-11-01,2620800,733824.0,3354624.0,Paid
3,INV-697615,SO-745068,C0382,HYD001,2021-10-09,2021-11-08,1094950,292961.0,1387911.0,Paid
4,INV-747829,SO-175713,C0472,CHN001,2024-11-04,2024-12-19,1250,225.0,1475.0,Paid


In [3]:
print("===INFO===")
print(invoices.info())

print("\n===Missing Values===")
print(invoices.isnull().sum())

print("\n===Sample Data===")
print(invoices.head())

===INFO===
<class 'pandas.DataFrame'>
RangeIndex: 18033 entries, 0 to 18032
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   invoice_id         18033 non-null  str    
 1   so_id              18033 non-null  str    
 2   customer_id        18033 non-null  str    
 3   branch_id          18033 non-null  str    
 4   invoice_date       18033 non-null  str    
 5   due_date           18033 non-null  str    
 6   total_order_value  18033 non-null  int64  
 7   total_gst_amount   18033 non-null  float64
 8   grand_total        18033 non-null  float64
 9   payment_status     18033 non-null  str    
dtypes: float64(2), int64(1), str(7)
memory usage: 1.4 MB
None

===Missing Values===
invoice_id           0
so_id                0
customer_id          0
branch_id            0
invoice_date         0
due_date             0
total_order_value    0
total_gst_amount     0
grand_total          0
payment_status       0


In [4]:
text_cols = [
    'invoice_id', 'so_id', 'customer_id',
    'branch_id', 'payment_status'
]

for col in text_cols:
    invoices[col] = invoices[col].str.upper()


In [5]:
invoices['invoice_date'] = pd.to_datetime(invoices['invoice_date'], errors='coerce')
invoices['due_date'] = pd.to_datetime(invoices['due_date'], errors='coerce')

invoices[['invoice_date', 'due_date']].head()


,invoice_date,due_date
0,2022-05-28,2022-06-27
1,2022-05-08,2022-06-07
2,2023-10-02,2023-11-01
3,2021-10-09,2021-11-08
4,2024-11-04,2024-12-19


In [8]:
invoices['days_to_due'] = (
    (invoices['due_date'] - invoices['invoice_date']).dt.days
)

invoices['days_overdue'] = (
    (pd.Timestamp.today() - invoices['due_date']).dt.days
)

invoices['overdue_flag'] = (
    (invoices['payment_status'] != 'PAID') &
    (invoices['days_overdue'] > 0)
).astype(int)

invoices['gst_pct'] = (
    invoices['total_gst_amount'] / invoices['total_order_value']
)


In [9]:
invoices.drop_duplicates(inplace=True)


In [10]:
print("=== DUPLICATE INVOICE IDs ===")
print(invoices['invoice_id'].duplicated().sum())

print("\n=== PAYMENT STATUS VALUES ===")
print(invoices['payment_status'].unique())

print("\n=== DATE CHECK ===")
print(invoices[['invoice_date', 'due_date']].head())

print("\n=== NUMERIC SUMMARY ===")
numeric_cols = [
    'total_order_value', 'total_gst_amount', 'grand_total'
]
print(invoices[numeric_cols].describe())


=== DUPLICATE INVOICE IDs ===
197

=== PAYMENT STATUS VALUES ===
<StringArray>
['UNPAID', 'PAID', 'PARTIALLY PAID']
Length: 3, dtype: str

=== DATE CHECK ===
  invoice_date   due_date
0   2022-05-28 2022-06-27
1   2022-05-08 2022-06-07
2   2023-10-02 2023-11-01
3   2021-10-09 2021-11-08
4   2024-11-04 2024-12-19

=== NUMERIC SUMMARY ===
       total_order_value  total_gst_amount   grand_total
count       1.803300e+04      1.803300e+04  1.803300e+04
mean        1.273165e+06      3.490826e+05  1.622248e+06
std         1.074819e+06      2.994372e+05  1.374170e+06
min         3.100000e+02      5.580000e+01  3.658000e+02
25%         4.260000e+05      1.145192e+05  5.405440e+05
50%         1.014840e+06      2.754914e+05  1.287381e+06
75%         1.842900e+06      5.069162e+05  2.348676e+06
max         8.169760e+06      2.282222e+06  1.045198e+07


In [12]:
invoices.to_csv('../data/processed/invoices.csv', index=False)
print("Processed invoices.csv saved to data/processed/")


Processed invoices.csv saved to data/processed/
